In [1]:
from google.colab import drive
drive.mount("/content/drive")

# CHANGE this to your repo folder
%cd /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
!ls

Mounted at /content/drive
/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
build_and_test.sh   LICENSE	    README.md	 SUPPORT.md
CODE_OF_CONDUCT.md  pipelines	    SECURITY.md  tests
experiments	    pyproject.toml  src		 test.sh


In [ ]:
!pip install -e .

In [ ]:
!pip install -q "peft==0.13.2" "transformers==4.41.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 10.3 MB/s eta 0:00:00


In [ ]:
!pip -q install -U "transformers>=4.45.0" "tokenizers>=0.20.0"


In [3]:
import os

BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(BASE_EVAL_DIR, exist_ok=True)

In [4]:
import logging
import sys

# Get the root logger or a named logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # allow INFO and above

for h in list(logger.handlers):
  logger.removeHandler(h)

# Create a handler that writes to stdout
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)

# (Optional) set a formatting for readability
formatter = logging.Formatter('%(levelname)s - %(message)s')
handler.setFormatter(formatter)

# Add handler to the logger
logger.addHandler(handler)

# Now test
logger.info("This will be printed to stdout")
logger.debug("This will not print (level is INFO)")

INFO - This will be printed to stdout


In [ ]:
!pip install evaluate

In [7]:
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm import tqdm

# ----------------------------
# 1. Config
# ----------------------------
MODEL_NAME = "Qwen/Qwen2-0.5B"   # <-- set your Qwen 0.6B model id here
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 16
LIMIT = 100

SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_PATH = os.path.join(SAVE_DIR, "results_qwen_0.6B_squadv1.json")

print("Device:", DEVICE)

# ----------------------------
# 2. Load model & tokenizer (CAUSAL LM!)
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# Qwen tokenizers often don't define pad_token; set it to eos for safe batching / generate()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# ----------------------------
# 3. Load SQuAD1.1 validation
# ----------------------------
dataset = load_dataset("squad", split="validation")
if LIMIT is not None:
    dataset = dataset.select(range(LIMIT))

print("Evaluating on", len(dataset), "examples")

# ----------------------------
# 4. Run generation
# ----------------------------
predictions = []
references = []

for doc in tqdm(dataset):
    prompt = (
        "Answer the question using the context.\n"
        "Context: " + doc["context"] +
        "\nQuestion: " + doc["question"] +
        "\nAnswer:"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,   # keep prompt bounded
        padding=False,
    )
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=4,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # For causal LM: remove the prompt tokens from the generated sequence
    gen_ids = out[0, prompt_len:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)

    # Keep only the first line
    text = text.split("\n", 1)[0].strip()

    # DEBUG: print first few
    if len(predictions) < 5:
        print("==== DEBUG EXAMPLE ====")
        print("Q   :", doc["question"])
        print("GT  :", doc["answers"]["text"])
        print("PRED:", repr(text))

    predictions.append({
        "id": doc["id"],
        "prediction_text": text,
    })
    references.append({
        "id": doc["id"],
        "answers": doc["answers"],
    })

# ----------------------------
# 5. Compute SQuAD1 metrics
# ----------------------------
metric = evaluate.load("squad")  # SQuAD 1.1 EM/F1
scores = metric.compute(predictions=predictions, references=references)
print("Scores:", scores)

# ----------------------------
# 6. Save results
# ----------------------------
with open(OUT_PATH, "w") as f:
    json.dump({
        "results": scores,
        "num_examples": len(dataset),
        "model": MODEL_NAME,
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_beams": 4,
    }, f, indent=2)

print("Saved to:", OUT_PATH)


Device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

==== DEBUG EXAMPLE ====
Q   : Which NFL team won Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'Denver Broncos'


  5%|▌         | 5/100 [00:07<01:27,  1.09it/s]

==== DEBUG EXAMPLE ====
Q   : What color was used to emphasize the 50th anniversary of the Super Bowl?
GT  : ['gold', 'gold', 'gold']
PRED: 'The color used to emphasize the 50th anniversary of the Super Bowl was'


100%|██████████| 100/100 [00:41<00:00,  2.40it/s]


Scores: {'exact_match': 33.0, 'f1': 49.02549117549116}
Saved to: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/results_qwen_0.6B_squadv1.json
